# GCS Ingestion Verification

This notebook demonstrates the new GCS ingestion flow, including identity extraction and credential switching.

In [ ]:
import sys
import os
sys.path.append("../..")

import asyncio
from mcp_servers.gcs.app.mcp_server import upload_object
from mcp_servers.gcs.app.schemas import UploadObjectRequest
from mcp_servers.gcs.app.config import GCS_SERVER_CONFIG
from unittest.mock import patch

# Mocking the context to simulate an authenticated request
mock_token = "mock-user-token"
mock_email = "user@example.com"

async def verify_ingestion():
    # We mock the external GCS calls and identity extraction for demonstration
    with patch("mcp_servers.gcs.app.mcp_server._get_current_token", return_value=mock_token), \
         patch("mcp_servers.gcs.app.mcp_server._fetch_user_email", return_value=mock_email), \
         patch("mcp_servers.gcs.app.mcp_server._make_sa_gcs_manager") as mock_sa, \
         patch("mcp_servers.gcs.app.mcp_server._make_gcs_manager") as mock_user:
        
        # Setup mock behavior
        mock_sa.return_value.copy_object.return_value.name = "ingested_report.pdf"
        
        print(f"--- Case 1: Internal Ingestion (KB Landing Zone) ---")
        request = UploadObjectRequest(
            source_uri="gs://ai_agent_landing_zone/user123/original.pdf",
            destination_bucket=GCS_SERVER_CONFIG.kb_landing_zone,
            name_of_the_file="ingested_report"
        )
        
        response = await upload_object(request)
        print(f"Status: {response.execution_status}")
        print(f"Message: {response.execution_message}")
        print(f"New URI: {response.gcs_uri}")
        
        # Verify SA was used
        print(f"Used SA Manager: {mock_sa.called}")
        
        print(f"\n--- Case 2: External Ingestion (User Bucket) ---")
        mock_user.return_value.copy_object.return_value.name = "my_copy.pdf"
        request_user = UploadObjectRequest(
            source_uri="gs://ai_agent_landing_zone/user123/original.pdf",
            destination_bucket="user-personal-bucket",
            metadata={"project": "Research"}
        )
        
        response_user = await upload_object(request_user)
        print(f"Status: {response_user.execution_status}")
        print(f"New URI: {response_user.gcs_uri}")
        print(f"Used OAuth Manager: {mock_user.called}")

await verify_ingestion()